# Project 3: SQL Data Analysis
### DecodeLabs - Data Analytics Industrial Training Kit | Batch 2026

---

> **Goal:** Use SQL queries to extract actionable business intelligence from the e-commerce dataset.

**Dataset:** `Project1_Cleaned_Dataset.xlsx` loaded into SQLite  
**Key Skills:** SELECT, WHERE, ORDER BY, GROUP BY, COUNT, SUM, AVG

## Step 1: Import Libraries & Set Up the SQL Environment
We use Python's built-in `sqlite3` to create an in-memory relational database and load our Excel data into it as a SQL table.

In [ ]:
# Standard libraries
import pandas as pd          # Read Excel, DataFrame operations
import sqlite3               # Built-in Python SQL engine (SQLite)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13, 'axes.labelsize': 11})

print('Libraries imported!')
print('We use SQLite: a lightweight, serverless SQL database built into Python.')
print('Every query written here is standard SQL that works on MySQL, PostgreSQL too.')

## Step 2: Load Data into SQLite Database
We read the Excel file with Pandas, then push it into an in-memory SQLite database using `to_sql()`. This creates a table called `orders` that we can query like any real database.

In [ ]:
# Step 2a: Read Excel into Pandas DataFrame
df = pd.read_excel('Project1_Cleaned_Dataset.xlsx')
df['Date'] = pd.to_datetime(df['Date'])  # parse dates properly

print(f'Excel loaded: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

df.head()

In [ ]:
# Step 2b: Create SQLite in-memory database
# sqlite3.connect(':memory:') creates a temporary database that lives in RAM
# It disappears when the Python session ends - perfect for practice
conn = sqlite3.connect(':memory:')

# Step 2c: Write the DataFrame into a SQL table called 'orders'
# to_sql() converts every row into an INSERT statement behind the scenes
# if_exists='replace' drops and recreates the table if it already exists
# index=False means do not add a pandas row-number column
df.to_sql('orders', conn, if_exists='replace', index=False)

print("Table 'orders' created inside SQLite!")
print("It is equivalent to: CREATE TABLE orders (OrderID TEXT, Date TEXT, ...)")
print("followed by 1,200 INSERT statements.")

In [ ]:
# Helper function: run SQL and return a DataFrame
# pd.read_sql_query() runs the SQL string against our connection
# and automatically returns the result as a Pandas DataFrame

def run_sql(query, title=''):
    result = pd.read_sql_query(query, conn)
    if title:
        print(f'\n{"="*55}')
        print(f'  {title}')
        print(f'{"="*55}')
    return result

print('Helper function ready. We can now write SQL to query the orders table.')

## Step 3: Basic SELECT Queries
The `SELECT` statement chooses which columns to display. `SELECT *` means 'all columns'. `LIMIT` restricts how many rows are returned.

In [ ]:
# QUERY 3.1 - View the first 5 rows of the entire table
# SELECT *    : fetch every column
# FROM orders : from the 'orders' table
# LIMIT 5     : return only the first 5 rows (saves memory)

q1 = 'SELECT * FROM orders LIMIT 5'
run_sql(q1, 'Query 3.1 - First 5 Rows (SELECT *)')

In [ ]:
# QUERY 3.2 - Select only specific columns we need
# Best practice: never SELECT * in production queries!

q2 = '''
SELECT OrderID, Date, Product, Quantity, UnitPrice, TotalPrice, OrderStatus
FROM orders
LIMIT 10
'''
run_sql(q2, 'Query 3.2 - Selected Columns Only')

In [ ]:
# QUERY 3.3 - COUNT total number of records in the table
# COUNT(*) counts every row including NULLs
# AS TotalOrders gives the result column a clean alias name

q3 = 'SELECT COUNT(*) AS TotalOrders FROM orders'
run_sql(q3, 'Query 3.3 - Total Row Count')

## Step 4: WHERE Clause - Filtering Rows
The `WHERE` clause is a row-level funnel. It evaluates every row and keeps only those matching the condition. **WHERE runs BEFORE SELECT** in the execution order.

In [ ]:
# QUERY 4.1 - Equality Filter: only 'Laptop' orders
# WHERE Product = 'Laptop' keeps only rows where Product equals 'Laptop'
# String values must be in single quotes in SQL

q4_1 = '''
SELECT OrderID, Date, Product, Quantity, TotalPrice, OrderStatus
FROM orders
WHERE Product = 'Laptop'
LIMIT 8
'''
result = run_sql(q4_1, 'Query 4.1 - Equality Filter (Laptops Only)')
print(f'Rows returned: {len(result)}')
result

In [ ]:
# QUERY 4.2 - Comparison Filter: orders where TotalPrice > 2000
# WHERE TotalPrice > 2000  numeric comparison, no quotes needed

q4_2 = '''
SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice, PaymentMethod
FROM orders
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC
LIMIT 10
'''
result = run_sql(q4_2, 'Query 4.2 - High-Value Orders (TotalPrice > $2,000)')
print(f'Rows returned: {len(result)}')
result

In [ ]:
# QUERY 4.3 - AND / OR Compound Conditions
# AND: BOTH conditions must be true
# OR:  EITHER condition is enough

q4_3 = '''
SELECT OrderID, Product, Quantity, TotalPrice, OrderStatus, ReferralSource
FROM orders
WHERE (Product = 'Laptop' OR Product = 'Phone')
  AND Quantity >= 3
  AND OrderStatus = 'Delivered'
ORDER BY TotalPrice DESC
LIMIT 10
'''
result = run_sql(q4_3, 'Query 4.3 - AND/OR Filter (Delivered Laptops/Phones, Qty >= 3)')
print(f'Rows returned: {len(result)}')
result

In [ ]:
# QUERY 4.4 - Pattern Matching with LIKE
# LIKE uses wildcards: % = any number of characters, _ = exactly 1 character
# Find all orders referred via social media channels

q4_4 = '''
SELECT OrderID, Product, TotalPrice, ReferralSource, PaymentMethod
FROM orders
WHERE ReferralSource LIKE '%book%'
   OR ReferralSource LIKE 'Insta%'
ORDER BY TotalPrice DESC
LIMIT 10
'''
run_sql(q4_4, 'Query 4.4 - LIKE Pattern Match (Social Media Referrals)')

In [ ]:
# QUERY 4.5 - IN Operator (cleaner alternative to multiple ORs)
# IN ('val1','val2') is equivalent to col = 'val1' OR col = 'val2'

q4_5 = '''
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
WHERE OrderStatus IN ('Cancelled', 'Returned')
ORDER BY TotalPrice DESC
LIMIT 10
'''
run_sql(q4_5, 'Query 4.5 - IN Operator (Cancelled or Returned Orders)')

## Step 5: ORDER BY - Sorting Results
`ORDER BY` sorts the final output. It runs **last** in the execution order, so it can sort by alias names created in SELECT.

In [ ]:
# QUERY 5.1 - Top 10 most expensive orders (DESC = highest first)

q5_1 = '''
SELECT OrderID, Date, Product, Quantity, UnitPrice, TotalPrice
FROM orders
ORDER BY TotalPrice DESC
LIMIT 10
'''
run_sql(q5_1, 'Query 5.1 - Top 10 Highest-Value Orders (ORDER BY DESC)')

In [ ]:
# QUERY 5.2 - Multi-column sort
# First by Product (A to Z), then by TotalPrice (high to low)

q5_2 = '''
SELECT Product, OrderID, Quantity, UnitPrice, TotalPrice
FROM orders
ORDER BY Product ASC,      -- primary sort: product name A to Z
         TotalPrice DESC   -- secondary sort: highest revenue first within each product
LIMIT 12
'''
run_sql(q5_2, 'Query 5.2 - Multi-Column Sort (Product ASC, TotalPrice DESC)')

## Step 6: GROUP BY - Aggregating into Buckets
`GROUP BY` collapses many rows into one summary row per group. It turns 1,200 raw rows into meaningful category-level insights.

In [ ]:
# QUERY 6.1 - COUNT orders per Product
# GROUP BY Product creates one row per unique product
# COUNT(*) counts how many orders exist in each product group

q6_1 = '''
SELECT   Product,
         COUNT(*)           AS TotalOrders,
         COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders) AS PctOfOrders
FROM     orders
GROUP BY Product
ORDER BY TotalOrders DESC
'''
run_sql(q6_1, 'Query 6.1 - Order Count per Product (GROUP BY + COUNT)')

In [ ]:
# QUERY 6.2 - SUM and AVG revenue per Product
# SUM(TotalPrice) = total revenue per product
# AVG(TotalPrice) = average order value per product
# ROUND(..., 2)   = round to 2 decimal places

q6_2 = '''
SELECT   Product,
         COUNT(*)                        AS TotalOrders,
         ROUND(SUM(TotalPrice), 2)       AS TotalRevenue,
         ROUND(AVG(TotalPrice), 2)       AS AvgOrderValue,
         ROUND(MIN(TotalPrice), 2)       AS MinOrder,
         ROUND(MAX(TotalPrice), 2)       AS MaxOrder
FROM     orders
GROUP BY Product
ORDER BY TotalRevenue DESC
'''
run_sql(q6_2, 'Query 6.2 - Revenue Summary per Product (SUM, AVG, MIN, MAX)')

In [ ]:
# QUERY 6.3 - Revenue and order count by PaymentMethod

q6_3 = '''
SELECT   PaymentMethod,
         COUNT(*)                        AS Orders,
         ROUND(SUM(TotalPrice), 2)       AS TotalRevenue,
         ROUND(AVG(TotalPrice), 2)       AS AvgOrderValue
FROM     orders
GROUP BY PaymentMethod
ORDER BY TotalRevenue DESC
'''
run_sql(q6_3, 'Query 6.3 - Revenue by Payment Method')

In [ ]:
# QUERY 6.4 - Order Status breakdown

q6_4 = '''
SELECT   OrderStatus,
         COUNT(*)                                         AS OrderCount,
         ROUND(SUM(TotalPrice), 2)                        AS Revenue,
         ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 1) AS PctOfTotal
FROM     orders
GROUP BY OrderStatus
ORDER BY OrderCount DESC
'''
run_sql(q6_4, 'Query 6.4 - Order Status Summary')

In [ ]:
# QUERY 6.5 - Multi-column GROUP BY: Product x PaymentMethod
# GROUP BY two columns creates one row per unique combination

q6_5 = '''
SELECT   Product,
         PaymentMethod,
         COUNT(*)                   AS Orders,
         ROUND(SUM(TotalPrice), 2)  AS Revenue
FROM     orders
GROUP BY Product, PaymentMethod
ORDER BY Product, Revenue DESC
'''
run_sql(q6_5, 'Query 6.5 - Cross-Tab: Product x Payment Method')

## Step 7: HAVING Clause - Filtering Aggregated Groups
`WHERE` filters individual rows. `HAVING` filters the groups **after** GROUP BY is applied. This is a common point of confusion — remember the execution order!

In [ ]:
# QUERY 7.1 - Products with more than 160 orders
# HAVING filters GROUPS (aggregated results)
# WHERE  filters ROWS  (individual records)

q7_1 = '''
SELECT   Product,
         COUNT(*)  AS TotalOrders
FROM     orders
GROUP BY Product
HAVING   COUNT(*) > 160
ORDER BY TotalOrders DESC
'''
run_sql(q7_1, 'Query 7.1 - HAVING: Products with > 160 Orders')

In [ ]:
# QUERY 7.2 - Referral channels where average order value > $1,000
# Combining WHERE (row filter) + GROUP BY + HAVING (group filter)

q7_2 = '''
SELECT   ReferralSource,
         COUNT(*)                   AS TotalOrders,
         ROUND(AVG(TotalPrice), 2)  AS AvgOrderValue,
         ROUND(SUM(TotalPrice), 2)  AS TotalRevenue
FROM     orders
WHERE    OrderStatus = 'Delivered'
GROUP BY ReferralSource
HAVING   AVG(TotalPrice) > 1000
ORDER BY AvgOrderValue DESC
'''
run_sql(q7_2, 'Query 7.2 - HAVING: High-Value Channels (Delivered Orders Only)')

## Step 8: Date-Based Analysis - Time Intelligence in SQL
We extract year and month from the Date column to perform time-series aggregations.

In [ ]:
# QUERY 8.1 - Revenue by Year
# strftime('%Y', Date) extracts 4-digit year from a date string
# SQLite syntax; MySQL uses YEAR(Date), PostgreSQL uses EXTRACT(YEAR FROM Date)

q8_1 = '''
SELECT   strftime('%Y', Date)          AS Year,
         COUNT(*)                       AS Orders,
         ROUND(SUM(TotalPrice), 2)      AS TotalRevenue,
         ROUND(AVG(TotalPrice), 2)      AS AvgOrderValue
FROM     orders
GROUP BY strftime('%Y', Date)
ORDER BY Year
'''
run_sql(q8_1, 'Query 8.1 - Yearly Revenue Summary')

In [ ]:
# QUERY 8.2 - Monthly Revenue (most recent 12 months)
# strftime('%Y-%m', Date) gives 'YYYY-MM' e.g. '2024-03'
# Text sort works here because of the YYYY-MM format

q8_2 = '''
SELECT   strftime('%Y-%m', Date)        AS YearMonth,
         COUNT(*)                        AS Orders,
         ROUND(SUM(TotalPrice),  2)      AS MonthlyRevenue,
         ROUND(AVG(TotalPrice),  2)      AS AvgOrderValue
FROM     orders
GROUP BY strftime('%Y-%m', Date)
ORDER BY YearMonth DESC
LIMIT 12
'''
run_sql(q8_2, 'Query 8.2 - Last 12 Months of Revenue')

In [ ]:
# QUERY 8.3 - Monthly revenue per Product (top 20 rows)

q8_3 = '''
SELECT   Product,
         strftime('%Y-%m', Date)         AS Month,
         ROUND(SUM(TotalPrice), 2)       AS Revenue
FROM     orders
GROUP BY Product, strftime('%Y-%m', Date)
ORDER BY Product, Revenue DESC
LIMIT 20
'''
run_sql(q8_3, 'Query 8.3 - Monthly Revenue per Product (Top 20 rows)')

## Step 9: Advanced Aggregations - Business KPIs
Complex queries that combine multiple SQL techniques to answer real business questions.

In [ ]:
# QUERY 9.1 - Cancellation Rate & Return Rate per Product
# CASE WHEN inside SUM acts as a conditional count:
# SUM(CASE WHEN condition THEN 1 ELSE 0 END) counts rows matching the condition

q9_1 = '''
SELECT
    Product,
    COUNT(*) AS TotalOrders,
    SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) AS Cancelled,
    SUM(CASE WHEN OrderStatus = 'Returned'  THEN 1 ELSE 0 END) AS Returned,
    ROUND(SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1.0 ELSE 0 END)
          / COUNT(*) * 100, 1)  AS CancelRate_Pct,
    ROUND(SUM(CASE WHEN OrderStatus = 'Returned'  THEN 1.0 ELSE 0 END)
          / COUNT(*) * 100, 1)  AS ReturnRate_Pct
FROM   orders
GROUP BY Product
ORDER BY CancelRate_Pct DESC
'''
run_sql(q9_1, 'Query 9.1 - Cancellation & Return Rates per Product')

In [ ]:
# QUERY 9.2 - Price Mismatch Report (Data Quality Check)
# PriceCheck = 'Mismatch' means TotalPrice is not equal to Quantity x UnitPrice

q9_2 = '''
SELECT
    Product,
    COUNT(*) AS TotalOrders,
    SUM(CASE WHEN PriceCheck = 'Mismatch' THEN 1 ELSE 0 END) AS Mismatches,
    ROUND(SUM(CASE WHEN PriceCheck = 'Mismatch' THEN 1.0 ELSE 0 END)
          / COUNT(*) * 100, 1)  AS MismatchRate_Pct
FROM   orders
GROUP BY Product
ORDER BY MismatchRate_Pct DESC
'''
run_sql(q9_2, 'Query 9.2 - Price Mismatch Rate by Product (Data Quality)')

In [ ]:
# QUERY 9.3 - Revenue Contribution % per Referral Channel
# A subquery (SELECT SUM...) inside the main query calculates the grand total

q9_3 = '''
SELECT
    ReferralSource,
    COUNT(*) AS Orders,
    ROUND(SUM(TotalPrice), 2) AS Revenue,
    ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS RevenuePct
FROM   orders
GROUP BY ReferralSource
ORDER BY Revenue DESC
'''
run_sql(q9_3, 'Query 9.3 - Revenue Contribution % by Referral Channel')

In [ ]:
# QUERY 9.4 - Top 10 Delivered Orders by Revenue (Final Business Report)

q9_4 = '''
SELECT
    OrderID, Date, CustomerID, Product,
    Quantity, UnitPrice, TotalPrice,
    PaymentMethod, OrderStatus, ReferralSource
FROM   orders
WHERE  OrderStatus = 'Delivered'
ORDER BY TotalPrice DESC
LIMIT 10
'''
run_sql(q9_4, 'Query 9.4 - Top 10 Delivered Orders by Revenue')

## Step 10: Visualising SQL Query Results
We run SQL queries and pipe results directly into Matplotlib/Seaborn charts - exactly how real BI tools work.

In [ ]:
# Fetch all data we need via SQL
revenue_by_product = pd.read_sql_query("""
    SELECT Product, ROUND(SUM(TotalPrice), 2) AS TotalRevenue
    FROM orders GROUP BY Product ORDER BY TotalRevenue DESC
""", conn)

status_counts = pd.read_sql_query("""
    SELECT OrderStatus, COUNT(*) AS OrderCount
    FROM orders GROUP BY OrderStatus ORDER BY OrderCount DESC
""", conn)

monthly_rev = pd.read_sql_query("""
    SELECT strftime('%Y-%m', Date) AS Month, ROUND(SUM(TotalPrice), 2) AS Revenue
    FROM orders GROUP BY Month ORDER BY Month
""", conn)

payment_avg = pd.read_sql_query("""
    SELECT PaymentMethod, ROUND(AVG(TotalPrice), 2) AS AvgValue
    FROM orders GROUP BY PaymentMethod ORDER BY AvgValue DESC
""", conn)

# Build 2x2 dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Step 10 - SQL Query Results: Business Dashboard', fontsize=15, fontweight='bold', y=1.01)

# Chart 1 - Revenue by Product
bars = axes[0,0].bar(revenue_by_product['Product'], revenue_by_product['TotalRevenue'], color='steelblue', edgecolor='white')
for b in bars:
    axes[0,0].text(b.get_x()+b.get_width()/2, b.get_height()+200, f'${b.get_height()/1000:.0f}K', ha='center', fontsize=8, fontweight='bold')
axes[0,0].set_title('Total Revenue by Product')
axes[0,0].set_ylabel('Revenue ($)')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# Chart 2 - Order Status Pie
axes[0,1].pie(status_counts['OrderCount'], labels=status_counts['OrderStatus'], autopct='%1.1f%%',
               colors=sns.color_palette('Set2', len(status_counts)), startangle=90, explode=[0.05]*len(status_counts))
axes[0,1].set_title('Order Status Distribution')

# Chart 3 - Monthly Revenue Trend
axes[1,0].plot(monthly_rev['Month'], monthly_rev['Revenue'], color='steelblue', marker='o', markersize=4, linewidth=2)
axes[1,0].fill_between(monthly_rev['Month'], monthly_rev['Revenue'], alpha=0.15, color='steelblue')
axes[1,0].set_title('Monthly Revenue Trend')
axes[1,0].set_xlabel('Month')
axes[1,0].set_ylabel('Revenue ($)')
axes[1,0].tick_params(axis='x', rotation=60, labelsize=7)
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# Chart 4 - Avg Order Value by Payment
bars4 = axes[1,1].bar(payment_avg['PaymentMethod'], payment_avg['AvgValue'], color='darkorange', edgecolor='white')
for b in bars4:
    axes[1,1].text(b.get_x()+b.get_width()/2, b.get_height()+5, f'${b.get_height():,.0f}', ha='center', fontsize=9, fontweight='bold')
axes[1,1].set_title('Avg Order Value by Payment Method')
axes[1,1].set_ylabel('Avg Total Price ($)')
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('step10_sql_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()
print('Dashboard saved!')

## Step 11: SQL Execution Order Recap
The order you **write** SQL clauses is NOT the order the database **executes** them.

In [ ]:
# Print the execution order with explanations
execution_order = [
    (1, 'FROM / JOIN', 'Identifies which table(s) to read data from'),
    (2, 'WHERE',       'Filters individual rows BEFORE any grouping'),
    (3, 'GROUP BY',    'Collapses rows into categorical buckets'),
    (4, 'HAVING',      'Filters buckets AFTER GROUP BY is applied'),
    (5, 'SELECT',      'Chooses columns and computes aliases/expressions'),
    (6, 'ORDER BY',    'Sorts the final result (can use aliases from SELECT)'),
]

print('SQL LOGICAL EXECUTION ORDER')
print('-' * 60)
for step, clause, desc in execution_order:
    print(f'  Step {step}: {clause:<12}  ->  {desc}')
print('-' * 60)

print()
print('THE ALIAS TRAP (Common Mistake):')
print('  WRONG:  SELECT TotalPrice AS tp  FROM orders  WHERE tp > 500')
print('  RIGHT:  SELECT TotalPrice AS tp  FROM orders  WHERE TotalPrice > 500')
print()
print('  Why? WHERE runs at Step 2, but alias tp is only created at Step 5.')
print('  The engine does not know what tp is yet when it evaluates WHERE!')

## Conclusion - Project 3 Complete!

In [ ]:
# Final comprehensive business summary via SQL
final = pd.read_sql_query("""
SELECT 'Total Orders'          AS Metric, CAST(COUNT(*) AS TEXT) AS Value FROM orders
UNION ALL SELECT 'Total Revenue', ROUND(SUM(TotalPrice),2) FROM orders
UNION ALL SELECT 'Avg Order Value', ROUND(AVG(TotalPrice),2) FROM orders
UNION ALL SELECT 'Median Order Value', TotalPrice FROM orders ORDER BY TotalPrice LIMIT 1 OFFSET (SELECT COUNT(*)/2 FROM orders)
""", conn)

# Simpler final report
results = pd.read_sql_query("""
SELECT
    COUNT(*) AS TotalOrders,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
    ROUND(MIN(TotalPrice), 2) AS MinOrder,
    ROUND(MAX(TotalPrice), 2) AS MaxOrder
FROM orders
""", conn)

print('PROJECT 3 - FINAL SQL BUSINESS REPORT')
print('=' * 50)
for col in results.columns:
    print(f'  {col:<20} : {results[col].values[0]}')
print('=' * 50)

# Close the database connection
conn.close()
print('SQLite connection closed. Project 3 complete!')

## SQL Concepts Mastered in Project 3

| Concept | Queries | Business Purpose |
|---------|---------|------------------|
| `SELECT` | 3.1, 3.2, 3.3 | Choose columns and compute metrics |
| `WHERE` | 4.1 - 4.5 | Filter rows: equality, comparison, LIKE, IN |
| `ORDER BY` | 5.1, 5.2 | Sort results ASC / DESC, multi-column |
| `GROUP BY` | 6.1 - 6.5 | Collapse rows into category summaries |
| `COUNT, SUM, AVG` | 6.1 - 6.5 | Aggregate functions for KPIs |
| `HAVING` | 7.1, 7.2 | Filter groups after aggregation |
| `strftime` | 8.1 - 8.3 | Extract year/month for time-series |
| `CASE WHEN` | 9.1, 9.2 | Conditional counting inside aggregations |
| Subqueries | 9.3, 9.4 | Calculate % contribution, nested filters |
| SQL to Chart | 10 | Pipe query results into Matplotlib visuals |

> **Key Insight:** SQL is declarative - you describe *what* you want, the database engine decides *how* to fetch it efficiently.